In [10]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")
sys.path.append('..')

from label2id import label2id_dict

from utils import *

In [11]:
# Load model directly
from transformers import AutoImageProcessor, AutoModelForImageClassification, pipeline
from datasets import load_dataset

import tqdm

import torch
from torch.utils.data import DataLoader

processor = AutoImageProcessor.from_pretrained("Ahmed9275/Vit-Cifar100", use_fast=True)
model = AutoModelForImageClassification.from_pretrained("Ahmed9275/Vit-Cifar100")
model.eval()

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=7

In [12]:
print(model)

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=7

In [13]:
model = model_obfuscation(model, noise_mag=1)

Obfuscating: vit.encoder.layer.0.attention.attention.query
Obfuscating: vit.encoder.layer.0.attention.attention.key
Obfuscating: vit.encoder.layer.0.attention.attention.value
Obfuscating: vit.encoder.layer.0.attention.output.dense
Obfuscating: vit.encoder.layer.0.intermediate.dense
Obfuscating: vit.encoder.layer.0.output.dense
Obfuscating: vit.encoder.layer.1.attention.attention.query
Obfuscating: vit.encoder.layer.1.attention.attention.key
Obfuscating: vit.encoder.layer.1.attention.attention.value
Obfuscating: vit.encoder.layer.1.attention.output.dense
Obfuscating: vit.encoder.layer.1.intermediate.dense
Obfuscating: vit.encoder.layer.1.output.dense
Obfuscating: vit.encoder.layer.2.attention.attention.query
Obfuscating: vit.encoder.layer.2.attention.attention.key
Obfuscating: vit.encoder.layer.2.attention.attention.value
Obfuscating: vit.encoder.layer.2.attention.output.dense
Obfuscating: vit.encoder.layer.2.intermediate.dense
Obfuscating: vit.encoder.layer.2.output.dense
Obfuscating: 

In [14]:
classifier = pipeline("image-classification", model=model, image_processor=processor, device='cuda')

In [15]:
dataset = load_dataset("cifar100")['test']
print(dataset)

Dataset({
    features: ['img', 'fine_label', 'coarse_label'],
    num_rows: 10000
})


In [16]:
# examples from training set

img = dataset['img'][0] # mountain 49

print(label2id_dict[classifier(img)[0]['label']])
print(dataset[0]['fine_label'])

img = dataset['img'][1] # forest 33

print(label2id_dict[classifier(img)[0]['label']])
print(dataset[1]['fine_label'])

49
49
33
33


In [17]:
# 定义预处理函数
def process_dataset(example_batch):
    # Take a list of PIL images and turn them to pixel values
    inputs = processor([x.convert('RGB') for x in example_batch['img']], return_tensors='pt')

    # Don't forget to include the labels!
    inputs['fine_label'] = example_batch['fine_label']
    return inputs

dataset.set_transform(process_dataset)


In [18]:
batch_size = 128  # You can adjust this based on your GPU memory
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

correct = 0
total = 0

model = model.to('cuda')

with torch.no_grad():  # Disable gradient calculation for evaluation
    for batch in tqdm.tqdm(dataloader):
        pixel_values = batch['pixel_values'].to('cuda')
        labels = batch['fine_label'].to('cuda')
        
        # Process images in batch
        batch_results = model(pixel_values=pixel_values)  # Assuming your classifier can handle batches
        
        predicted_labels = batch_results.logits.argmax(dim=-1)
        
        # Compare with ground truth
        correct += (predicted_labels == labels).sum().item()
        total += len(labels)

accuracy = correct / total
print(f"Correct: {correct}, Total: {total}, Accuracy: {accuracy:7f}")

100%|██████████| 79/79 [00:24<00:00,  3.16it/s]

Correct: 8985, Total: 10000, Accuracy: 0.898500
